# Deterministic Benchmark Sweep Notebook

This notebook runs deterministic-only benchmark comparisons for a selected checkpoint set.

Scope:
1. restore a saved training run
2. rebuild the evaluation dataset from metadata
3. benchmark selected checkpoints against equal-weight and SPY
4. summarize which checkpoints beat both baselines across all horizons

This notebook does not run stochastic regime sections.


In [ ]:
import copy
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

GIT_REPO_URL = "https://github.com/Dave-DKings/tcn_tape_vectorized_version.git"
GIT_BRANCH = "feature/run17-film-drive-org-20260318"
CLONE_IF_MISSING = True
CLONE_PARENT_DIR = Path('/content')
CLONE_DIR_NAME = 'tcn_tape_vectorized_version_clean'
INSTALL_REQUIREMENTS = False
AUTO_INSTALL_MISSING_REQUIREMENTS = True

CRITICAL_RUNTIME_MODULES = {
    'pandas_ta_classic': 'pandas-ta-classic>=0.3.59',
    'fredapi': 'fredapi>=0.5.1',
    'yfinance': 'yfinance>=0.2.38',
}


def run(cmd):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(cmd, check=True)


def missing_runtime_requirements() -> list[str]:
    missing = []
    for module_name, requirement in CRITICAL_RUNTIME_MODULES.items():
        if importlib.util.find_spec(module_name) is None:
            missing.append(requirement)
    return missing


def normalize_github_url(url: str | None) -> str | None:
    if not url:
        return url
    url = str(url).strip()
    if url.startswith('git@github.com:'):
        repo = url[len('git@github.com:'):]
        if repo.endswith('.git'):
            repo = repo[:-4]
        return f'https://github.com/{repo}.git'
    return url


def safe_cwd() -> Path | None:
    try:
        return Path.cwd().resolve()
    except FileNotFoundError:
        return None


def safe_resolve(path_like) -> Path | None:
    p = Path(path_like)
    try:
        if p.is_absolute():
            return p.resolve()
    except Exception:
        return None
    cwd = safe_cwd()
    if cwd is None:
        return None
    try:
        return (cwd / p).resolve()
    except Exception:
        return None


def find_repo_root() -> Path | None:
    candidate_roots = []
    seen = set()
    cwd = safe_cwd()
    if cwd is not None:
        for p in [cwd, *cwd.parents]:
            rp = safe_resolve(p)
            if rp is not None and rp not in seen:
                seen.add(rp)
                candidate_roots.append(rp)
    for p in [
        Path(globals().get('EVAL_REPO_DIR', CLONE_PARENT_DIR / CLONE_DIR_NAME)),
        CLONE_PARENT_DIR / CLONE_DIR_NAME,
        Path('/content/tcn_tape_vectorized_version_clean'),
        Path('/mnt/c/Users/Owner/tcn_tape_vectorized_version_clean'),
    ]:
        rp = safe_resolve(p)
        if rp is not None and rp not in seen:
            seen.add(rp)
            candidate_roots.append(rp)
    for p in candidate_roots:
        if (p / 'src' / 'config.py').exists():
            return p
    return None


REPO_ROOT = find_repo_root()
GIT_REPO_URL = normalize_github_url(GIT_REPO_URL)

if REPO_ROOT is None and CLONE_IF_MISSING:
    if not GIT_REPO_URL:
        raise FileNotFoundError('Repo root not found and GIT_REPO_URL is not set.')
    CLONE_PARENT_DIR.mkdir(parents=True, exist_ok=True)
    clone_target = CLONE_PARENT_DIR / CLONE_DIR_NAME
    if clone_target.exists() and not (clone_target / '.git').exists():
        shutil.rmtree(clone_target, ignore_errors=True)
    if not clone_target.exists():
        clone_cmd = ['git', 'clone', GIT_REPO_URL, str(clone_target)]
        print('+', ' '.join(map(str, clone_cmd)))
        clone_proc = subprocess.run(clone_cmd, text=True, capture_output=True)
        if clone_proc.stdout:
            print(clone_proc.stdout, end='')
        if clone_proc.returncode != 0:
            stderr = (clone_proc.stderr or '').strip()
            raise RuntimeError(f'git clone failed: {stderr}')
    REPO_ROOT = clone_target.resolve()

if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate repo root containing src/config.py.')

if GIT_BRANCH and (REPO_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin'], check=False)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', GIT_BRANCH], check=False)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

requirements_file = REPO_ROOT / 'requirements.txt'
missing_requirements = missing_runtime_requirements()
should_install = INSTALL_REQUIREMENTS or (AUTO_INSTALL_MISSING_REQUIREMENTS and bool(missing_requirements))
if should_install:
    run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
    if INSTALL_REQUIREMENTS and requirements_file.exists():
        run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)])
    elif missing_requirements:
        run([sys.executable, '-m', 'pip', 'install', *missing_requirements])

from src.config import build_run21_config
from src.notebook_helpers.tcn_phase1 import (
    Phase1Dataset,
    compare_agent_vs_baseline,
    create_experiment6_result_stub,
    evaluate_experiment6_checkpoint,
    load_training_metadata_into_config,
    prepare_phase1_dataset,
)

print('REPO_ROOT:', REPO_ROOT)
print('Missing critical requirements before install:', missing_requirements)
print('Dependency install executed:', should_install)


In [ ]:
RUN_ID = 'run21_top10'
CHECKPOINT_SELECTIONS = [
    {'episode': 413, 'checkpoint_kind': 'high_watermark'},
    {'episode': 438, 'checkpoint_kind': 'high_watermark'},
    {'episode': 461, 'checkpoint_kind': 'high_watermark'},
    {'episode': 512, 'checkpoint_kind': 'high_watermark'},
    {'episode': 390, 'checkpoint_kind': 'high_watermark'},
    {'episode': 446, 'checkpoint_kind': 'high_watermark'},
    {'episode': 473, 'checkpoint_kind': 'high_watermark'},
    {'episode': 528, 'checkpoint_kind': 'high_watermark'},
    {'episode': 482, 'checkpoint_kind': 'high_watermark'},
    {'episode': 568, 'checkpoint_kind': 'high_watermark'},
]


ASSET_TICKERS_OVERRIDE = [
    'TSLA', 'META', 'NVDA', 'AAPL', 'MSFT',
    'AMZN', 'GOOGL', 'LLY', 'JPM', 'WMT',
]
EVAL_METADATA_PATH_OVERRIDE = None
EVAL_RANDOM_SEED = 42
EVAL_FORCE_TEST_START_DATE = '2020-01-01'
AUTO_RESUME_LATEST_EVAL_SESSION = True
RESUME_EVAL_SESSION_TAG = None
FORCE_NEW_EVAL_SESSION = False
EVAL_DETERMINISTIC_MODE = 'mean'
STOCHASTIC_RANDOM_START = False

HORIZON_YEARS = [1, 2, 3, 4]
HORIZON_DAYS = {year: 252 * year for year in HORIZON_YEARS}
BENCHMARK_START_OFFSETS = [0, 63, 126]
CVAR_ALPHA = 0.05
BENCHMARK_SCHEMA_VERSION = 1
BENCHMARK_RISK_FREE_RATE = 0.02
SAVE_EVAL_LOGS = True
SAVE_EVAL_ARTIFACTS = True
RUN_LIMIT_BENCHMARK = None

AUTO_RESTORE_RESULTS_FROM_DRIVE = True
FORCE_RESTORE_RESULTS = False
RUN_RESULTS_ZIP_PATH = Path('/content/drive/MyDrive/tcn_tape_vectorized_runs/run21_top10/tcn_tape_vectorized_run21_top10.zip')
DRIVE_RUN_ROOT = Path('/content/drive/MyDrive/tcn_tape_vectorized_runs/run21_top10')
DRIVE_EVAL_PARENT = DRIVE_RUN_ROOT / 'deterministic_benchmark_evaluation'
EVAL_RESTORE_DIR = Path('/content/eval_restore')

print('RUN_ID:', RUN_ID)
print('Checkpoints:', CHECKPOINT_SELECTIONS)
print('Asset override:', ASSET_TICKERS_OVERRIDE)
print('Benchmark risk-free rate:', BENCHMARK_RISK_FREE_RATE)
print('Run zip:', RUN_RESULTS_ZIP_PATH)


In [ ]:
def _running_in_colab() -> bool:
    return importlib.util.find_spec('google.colab') is not None


def _maybe_mount_drive() -> None:
    if not _running_in_colab():
        return
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        return
    from google.colab import drive
    drive.mount('/content/drive')


def _has_metadata(results_root: Path) -> bool:
    logs_dir = results_root / 'logs'
    return logs_dir.exists() and any(logs_dir.glob('*_metadata.json'))


def _restore_results_from_drive_if_needed() -> None:
    if not AUTO_RESTORE_RESULTS_FROM_DRIVE:
        return
    if _running_in_colab():
        _maybe_mount_drive()
    if not RUN_RESULTS_ZIP_PATH.exists():
        print('[INFO] Run results zip not found on Drive:', RUN_RESULTS_ZIP_PATH)
        return
    current_root = EVAL_RESTORE_DIR / 'tcn_fusion_results'
    if current_root.exists() and _has_metadata(current_root) and not FORCE_RESTORE_RESULTS:
        print('[OK] Existing restored results found:', current_root)
        return
    if EVAL_RESTORE_DIR.exists():
        shutil.rmtree(EVAL_RESTORE_DIR, ignore_errors=True)
    EVAL_RESTORE_DIR.mkdir(parents=True, exist_ok=True)
    import zipfile
    with zipfile.ZipFile(RUN_RESULTS_ZIP_PATH, 'r') as zf:
        zf.extractall(EVAL_RESTORE_DIR)
    print('[OK] Restored results zip to:', EVAL_RESTORE_DIR)


def _find_named_dirs(base: Path, dir_name: str) -> list[Path]:
    if not base.exists():
        return []
    matches = []
    direct = base / dir_name
    if direct.exists() and direct.is_dir():
        matches.append(direct)
    try:
        for p in base.rglob(dir_name):
            if p.is_dir() and p not in matches:
                matches.append(p)
    except Exception:
        pass
    return matches


def _list_existing_eval_sessions(parent: Path) -> list[Path]:
    if not parent.exists():
        return []
    return sorted([p for p in parent.iterdir() if p.is_dir()])


def _session_sort_key(session_root: Path):
    preferred = [
        session_root / '02_manifest' / 'manifests' / 'benchmark_manifest.json',
        session_root / '00_metadata' / 'manifests' / 'session_setup.json',
    ]
    for p in preferred:
        if p.exists():
            return (p.stat().st_mtime, session_root.name)
    return (session_root.stat().st_mtime, session_root.name)


def _resolve_eval_session_root(eval_parent: Path) -> tuple[Path, str, bool]:
    eval_parent.mkdir(parents=True, exist_ok=True)
    default_tag = datetime.utcnow().strftime('det_benchmark_sweep_%Y%m%d_%H%M%S_utc')

    requested_tag = str(RESUME_EVAL_SESSION_TAG).strip() if RESUME_EVAL_SESSION_TAG else None
    if requested_tag:
        requested_root = eval_parent / requested_tag
        if requested_root.exists():
            return requested_root.resolve(), requested_root.name, True
        requested_root.mkdir(parents=True, exist_ok=True)
        return requested_root.resolve(), requested_root.name, False

    existing_sessions = sorted(_list_existing_eval_sessions(eval_parent), key=_session_sort_key, reverse=True)
    if AUTO_RESUME_LATEST_EVAL_SESSION and existing_sessions and not FORCE_NEW_EVAL_SESSION:
        chosen = existing_sessions[0]
        return chosen.resolve(), chosen.name, True

    new_root = eval_parent / default_tag
    new_root.mkdir(parents=True, exist_ok=True)
    return new_root.resolve(), new_root.name, False


_restore_results_from_drive_if_needed()

repo_dir = Path(str(REPO_ROOT))
restore_dir = EVAL_RESTORE_DIR

results_candidates = []
for root in [restore_dir, repo_dir]:
    for candidate in _find_named_dirs(root, 'tcn_fusion_results'):
        if candidate not in results_candidates:
            results_candidates.append(candidate)

metadata_candidates = [p for p in results_candidates if _has_metadata(p)]
if metadata_candidates:
    EVAL_RESULTS_ROOT = sorted(
        metadata_candidates,
        key=lambda p: max((f.stat().st_mtime for f in (p / 'logs').glob('*_metadata.json')), default=0),
        reverse=True,
    )[0]
elif results_candidates:
    EVAL_RESULTS_ROOT = results_candidates[0]
else:
    EVAL_RESULTS_ROOT = restore_dir / 'tcn_fusion_results'

prep_candidates = []
for root in [restore_dir, EVAL_RESULTS_ROOT.parent, repo_dir]:
    for candidate in _find_named_dirs(root, 'data_exports'):
        if candidate not in prep_candidates:
            prep_candidates.append(candidate)
EVAL_PREP_ARTIFACTS_DIR = next((p for p in prep_candidates if p.exists()), repo_dir / 'data_exports')

eval_parent = DRIVE_EVAL_PARENT if (_running_in_colab() or DRIVE_EVAL_PARENT.exists() or DRIVE_EVAL_PARENT.parent.exists()) else (EVAL_RESULTS_ROOT / 'deterministic_benchmark_evaluation')
EVAL_SESSION_ROOT, SESSION_TAG, EVAL_SESSION_RESUMED = _resolve_eval_session_root(eval_parent)
EVAL_SESSION_ROOT.mkdir(parents=True, exist_ok=True)

print('EVAL_RESULTS_ROOT:', EVAL_RESULTS_ROOT)
print('EVAL_PREP_ARTIFACTS_DIR:', EVAL_PREP_ARTIFACTS_DIR)
print('EVAL_SESSION_ROOT:', EVAL_SESSION_ROOT)
print('SESSION_TAG:', SESSION_TAG)
print('EVAL_SESSION_RESUMED:', EVAL_SESSION_RESUMED)


## Build Metadata-Aligned Evaluation Config and Dataset
This section rebuilds the dataset from the restored training metadata, then applies an explicit ticker override if needed.


In [ ]:
def apply_asset_universe_override(config: dict, tickers: list[str] | None) -> dict:
    cfg = copy.deepcopy(config)
    if not tickers:
        return cfg
    tickers = list(tickers)
    num_assets = len(tickers)
    cfg['ASSET_TICKERS'] = tickers
    cfg['NUM_ASSETS'] = num_assets
    for section_name in ['environment_params', 'agent_params', 'training_params', 'data_params']:
        section = cfg.get(section_name)
        if isinstance(section, dict):
            section['asset_tickers'] = list(tickers)
            section['num_assets'] = num_assets
    return cfg


def _find_metadata_files(search_roots: list[Path]) -> list[Path]:
    files = []
    for root in search_roots:
        logs_dir = root / 'logs'
        if logs_dir.exists():
            files.extend(logs_dir.glob('*_metadata.json'))
    unique = []
    seen = set()
    for p in sorted(files, key=lambda p: p.stat().st_mtime, reverse=True):
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            unique.append(rp)
    return unique


eval_config = build_run21_config('phase1')

metadata_search_roots = []
for root in [EVAL_RESULTS_ROOT, restore_dir / 'tcn_fusion_results', repo_dir / 'tcn_fusion_results']:
    if root not in metadata_search_roots:
        metadata_search_roots.append(root)

if EVAL_METADATA_PATH_OVERRIDE:
    EVAL_METADATA_PATH = Path(EVAL_METADATA_PATH_OVERRIDE).resolve()
else:
    meta_files = _find_metadata_files(metadata_search_roots)
    if not meta_files:
        searched = '\n'.join(str(root / 'logs') for root in metadata_search_roots)
        raise FileNotFoundError(f'No metadata JSON found. Searched:\n{searched}')
    matched_meta = []
    for meta_path in meta_files:
        try:
            meta = json.loads(meta_path.read_text(encoding='utf-8'))
            checkpointing = meta.get('Checkpointing', {}) or {}
            if str(checkpointing.get('high_watermark_checkpoint_subdir', '')).strip() == 'high_watermark_checkpoints_run21':
                matched_meta.append(meta_path)
        except Exception:
            pass
    EVAL_METADATA_PATH = matched_meta[0] if matched_meta else meta_files[0]

EVAL_METADATA_PAYLOAD = json.loads(Path(EVAL_METADATA_PATH).read_text(encoding='utf-8'))
print('Using metadata:', EVAL_METADATA_PATH)
load_training_metadata_into_config(EVAL_METADATA_PATH, eval_config)

run_ctx = EVAL_METADATA_PAYLOAD.get('Run_Context', {}) or {}
train_date_min = run_ctx.get('train_date_min')
test_date_min = run_ctx.get('test_date_min')
test_date_max = run_ctx.get('test_date_max')
if train_date_min:
    eval_config['ANALYSIS_START_DATE'] = str(pd.Timestamp(train_date_min).date())
if test_date_max:
    eval_config['ANALYSIS_END_DATE'] = str(pd.Timestamp(test_date_max).date())
if test_date_min:
    eval_config['TRAIN_TEST_SPLIT_DATE'] = str((pd.Timestamp(test_date_min) - pd.Timedelta(days=1)).date())
if EVAL_FORCE_TEST_START_DATE:
    forced_start = pd.Timestamp(EVAL_FORCE_TEST_START_DATE)
    eval_config['TRAIN_TEST_SPLIT_DATE'] = str((forced_start - pd.Timedelta(days=1)).date())

eval_config = apply_asset_universe_override(eval_config, ASSET_TICKERS_OVERRIDE)

EVAL_HW_SUBDIR = str(eval_config.get('training_params', {}).get('high_watermark_checkpoint_subdir', 'high_watermark_checkpoints_run21'))
EVAL_STEP_SUBDIR = str(eval_config.get('training_params', {}).get('step_sharpe_checkpoint_subdir', 'step_sharpe_checkpoints_run21'))

prep_dir = Path(EVAL_PREP_ARTIFACTS_DIR)
eval_phase1_data = prepare_phase1_dataset(
    eval_config,
    force_download=False,
    save_preparation_artifacts=False,
    preparation_artifacts_dir=str(prep_dir) if prep_dir.exists() else None,
)

if ASSET_TICKERS_OVERRIDE and list(eval_config['ASSET_TICKERS']) != list(ASSET_TICKERS_OVERRIDE):
    raise RuntimeError(f'Universe mismatch. Expected {ASSET_TICKERS_OVERRIDE}, got {eval_config["ASSET_TICKERS"]}')

print('Train shape:', eval_phase1_data.train_df.shape)
print('Test shape:', eval_phase1_data.test_df.shape)
print('Tickers:', eval_config['ASSET_TICKERS'])
print('Checkpoint subdirs:', EVAL_HW_SUBDIR, EVAL_STEP_SUBDIR)
print('Analysis window:', eval_config.get('ANALYSIS_START_DATE'), '->', eval_config.get('ANALYSIS_END_DATE'))
print('Train/test split date:', eval_config.get('TRAIN_TEST_SPLIT_DATE'))


## Session Directories and Benchmark Helpers
All outputs are written under a dedicated deterministic benchmark session root.


In [ ]:
def save_df(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return path


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    return path


def load_csv_or_empty(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


def load_json_or_none(path: Path):
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return None


def _drop_unnamed_cols(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame() if df is None else df
    keep_cols = [c for c in df.columns if not str(c).startswith('Unnamed:')]
    return df.loc[:, keep_cols].copy()


def stage_file_has_rows(path: Path, required_col: str | None = None) -> bool:
    try:
        df = _drop_unnamed_cols(load_csv_or_empty(path))
    except Exception:
        return False
    if df.empty:
        return False
    if required_col and required_col not in df.columns:
        return False
    return True


def make_stage_dirs(stage_name: str) -> dict:
    root = EVAL_SESSION_ROOT / stage_name
    dirs = {'root': root}
    for key in ['plans', 'tables', 'logs', 'tracks', 'weights', 'alphas', 'manifests']:
        dirs[key] = root / key
        dirs[key].mkdir(parents=True, exist_ok=True)
    return dirs


SECTION_DIRS = {
    'metadata': make_stage_dirs('00_metadata'),
    'benchmarks': make_stage_dirs('01_benchmarks'),
    'manifest': make_stage_dirs('02_manifest'),
}

ARTIFACT_COPY_SEEN = set()


def get_eval_artifact_sources() -> list[Path]:
    candidates = [
        EVAL_RESULTS_ROOT / EVAL_HW_SUBDIR / 'logs',
        EVAL_RESULTS_ROOT / EVAL_STEP_SUBDIR / 'logs',
        EVAL_RESULTS_ROOT / 'logs',
    ]
    out = []
    for p in candidates:
        if p.exists() and p not in out:
            out.append(p)
    return out


def _artifact_target_dir(stage_dirs: dict, file_name: str) -> Path:
    name = file_name.lower()
    if '_weights_' in name:
        return stage_dirs['weights']
    if '_alphas_' in name:
        return stage_dirs['alphas']
    if '_actions_' in name or '_portfolio_' in name or '_history_' in name:
        return stage_dirs['tracks']
    return stage_dirs['logs']


def copy_eval_outputs_since(start_ts: float, stage_dirs: dict, block_prefix: str) -> list[str]:
    copied = []
    for src_dir in get_eval_artifact_sources():
        for p in sorted(src_dir.glob('*')):
            if not p.is_file():
                continue
            if p.stat().st_mtime + 1e-6 < start_ts:
                continue
            key = (str(p.resolve()), p.stat().st_mtime_ns)
            if key in ARTIFACT_COPY_SEEN:
                continue
            ARTIFACT_COPY_SEEN.add(key)
            target_dir = _artifact_target_dir(stage_dirs, p.name)
            dst = target_dir / f'{block_prefix}__{p.name}'
            shutil.copy2(p, dst)
            copied.append(str(dst.relative_to(EVAL_SESSION_ROOT)))
    return copied


def discover_checkpoint_pairs(results_root: Path, *, high_watermark_subdir: str | None = None, step_sharpe_subdir: str | None = None, include_root: bool = False) -> pd.DataFrame:
    rows = []
    search_dirs = []
    if high_watermark_subdir:
        search_dirs.append((results_root / high_watermark_subdir, 'high_watermark'))
    if step_sharpe_subdir:
        search_dirs.append((results_root / step_sharpe_subdir, 'periodic_step'))
    if include_root:
        search_dirs.append((results_root, 'root'))
    for base_dir, kind in search_dirs:
        if not base_dir.exists():
            continue
        pattern = '*_actor.weights.h5' if kind == 'root' else '**/*_actor.weights.h5'
        for actor in sorted(base_dir.glob(pattern)):
            prefix = str(actor).replace('_actor.weights.h5', '')
            critic = Path(prefix + '_critic.weights.h5')
            if not critic.exists():
                continue
            name = actor.name
            m_sh = re.search(r'_sh([pm])(\d+)p(\d+)', name)
            sharpe_tag = None
            if m_sh:
                sign = 1.0 if m_sh.group(1) == 'p' else -1.0
                sharpe_tag = sign * float(f"{m_sh.group(2)}.{m_sh.group(3)}")
            m_ep = re.search(r'_ep(\d+)', name)
            m_st = re.search(r'_step(\d+)', name)
            rows.append({
                'checkpoint_prefix': prefix,
                'actor_path': str(actor),
                'critic_path': str(critic),
                'checkpoint_kind': kind,
                'episode': int(m_ep.group(1)) if m_ep else np.nan,
                'step': int(m_st.group(1)) if m_st else np.nan,
                'sharpe_tag': sharpe_tag,
                'mtime': actor.stat().st_mtime,
            })
    if not rows:
        raise RuntimeError(f'No valid actor+critic checkpoints under {results_root}')
    return pd.DataFrame(rows).sort_values(['checkpoint_kind', 'episode', 'step', 'mtime'], ascending=[True, True, True, False]).reset_index(drop=True)


def resolve_checkpoint_row(df_ckpt: pd.DataFrame, episode: int, checkpoint_kind: str) -> pd.Series:
    subset = df_ckpt[(df_ckpt['checkpoint_kind'] == checkpoint_kind) & (df_ckpt['episode'] == episode)].copy()
    if subset.empty:
        raise RuntimeError(f'Checkpoint ep{episode:04d} not found for kind={checkpoint_kind}')
    subset = subset.sort_values(['mtime', 'sharpe_tag'], ascending=[False, False]).reset_index(drop=True)
    return subset.iloc[0]


def make_phase1_slice(base_phase1: Phase1Dataset, start_offset: int, horizon_days: int):
    test_df = base_phase1.test_df.copy()
    test_df['Date'] = pd.to_datetime(test_df['Date'])
    unique_dates = pd.Series(test_df['Date'].dropna().unique()).sort_values().reset_index(drop=True)
    if start_offset >= len(unique_dates):
        return None, None
    end_idx = min(len(unique_dates), int(start_offset) + int(horizon_days))
    win_dates = unique_dates.iloc[int(start_offset):end_idx]
    if len(win_dates) < max(60, int(horizon_days * 0.5)):
        return None, None
    d0 = pd.to_datetime(win_dates.iloc[0])
    d1 = pd.to_datetime(win_dates.iloc[-1])
    sliced_df = test_df[(test_df['Date'] >= d0) & (test_df['Date'] <= d1)].copy()
    if sliced_df.empty:
        return None, None
    phase1_slice = copy.deepcopy(base_phase1)
    phase1_slice.test_df = sliced_df
    phase1_slice.test_start_date = d0
    return phase1_slice, {
        'window_start_date': str(d0.date()),
        'window_end_date': str(d1.date()),
        'n_days': int(len(win_dates)),
    }


def eval_run_one_checkpoint(eval_cfg, phase1_data, ckpt_prefix, *, seed, save_logs, save_artifacts):
    stub = create_experiment6_result_stub(
        random_seed=seed,
        use_covariance=True,
        architecture=eval_cfg['agent_params']['actor_critic_type'],
        checkpoint_path=ckpt_prefix,
        agent_config=copy.deepcopy(eval_cfg['agent_params']),
        base_agent_params=None,
    )
    return evaluate_experiment6_checkpoint(
        experiment6=stub,
        phase1_data=phase1_data,
        config=eval_cfg,
        random_seed=seed,
        checkpoint_path_override=ckpt_prefix,
        deterministic_eval_mode=EVAL_DETERMINISTIC_MODE,
        num_eval_runs=0,
        stochastic_eval_mode='sample',
        stochastic_episode_length_limit=int(len(pd.to_datetime(phase1_data.test_df['Date']).drop_duplicates())),
        stochastic_random_start=bool(STOCHASTIC_RANDOM_START),
        save_eval_logs=bool(save_logs),
        save_eval_artifacts=bool(save_artifacts),
    )


def compute_tail_risk_metrics(returns, alpha: float = CVAR_ALPHA) -> dict:
    series = pd.Series(returns, dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    if series.empty:
        return {
            'num_obs': 0,
            'mean_return': np.nan,
            'std_return': np.nan,
            'var_alpha': np.nan,
            'cvar_alpha': np.nan,
            'var_alpha_pct': np.nan,
            'cvar_alpha_pct': np.nan,
        }
    var_alpha = float(series.quantile(float(alpha)))
    tail = series[series <= var_alpha]
    cvar_alpha = float(tail.mean()) if not tail.empty else var_alpha
    return {
        'num_obs': int(series.shape[0]),
        'mean_return': float(series.mean()),
        'std_return': float(series.std(ddof=0)),
        'var_alpha': var_alpha,
        'cvar_alpha': cvar_alpha,
        'var_alpha_pct': var_alpha * 100.0,
        'cvar_alpha_pct': cvar_alpha * 100.0,
    }


def _find_raw_ohlcv_artifact() -> Path | None:
    search_roots = []
    for root in [globals().get('EVAL_PREP_ARTIFACTS_DIR'), REPO_ROOT / 'data_exports' if 'REPO_ROOT' in globals() else None, Path('data_exports')]:
        if root is None:
            continue
        p = Path(root)
        if p.exists() and p not in search_roots:
            search_roots.append(p)
    candidates = []
    for root in search_roots:
        patterns = ['*_raw_ohlcv.csv', '**/*_raw_ohlcv.csv']
        for pattern in patterns:
            for p in root.glob(pattern):
                if p.is_file():
                    candidates.append(p)
    if not candidates:
        return None
    candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def _validate_baseline_series(name: str, series: pd.Series) -> pd.Series:
    s = pd.Series(series, dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        raise ValueError(f'{name} baseline is empty')
    if float(s.abs().quantile(0.99)) > 0.25:
        raise ValueError(f'{name} baseline appears mis-scaled: 99th percentile abs return={float(s.abs().quantile(0.99)):.4f}')
    return s


def eval_fetch_spy_returns(start_date: pd.Timestamp, end_date: pd.Timestamp) -> pd.Series:
    try:
        import yfinance as yf
    except Exception:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'yfinance'])
            import yfinance as yf
        except Exception:
            print('[WARN] Could not install/import yfinance; SPY benchmark disabled.')
            return pd.Series(dtype=float)
    try:
        download_start = pd.Timestamp(start_date) - pd.Timedelta(days=7)
        df = yf.download('SPY', start=str(download_start.date()), end=str((end_date + pd.Timedelta(days=1)).date()), auto_adjust=True, progress=False)
        if df is None or df.empty:
            return pd.Series(dtype=float)
        close = df['Close']
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        ret = pd.to_numeric(close, errors='coerce').dropna().pct_change().dropna().astype(float)
        ret.index = pd.to_datetime(ret.index)
        return ret
    except Exception as e:
        print(f'[WARN] SPY fetch failed: {type(e).__name__}: {e}')
        return pd.Series(dtype=float)


def build_baselines_from_phase1(phase1_data: Phase1Dataset):
    test_df = phase1_data.test_df.copy()
    if 'Date' not in test_df.columns or 'Ticker' not in test_df.columns:
        raise ValueError('test_df must contain Date and Ticker columns')

    raw_ohlcv_path = _find_raw_ohlcv_artifact()
    if raw_ohlcv_path is None:
        raise FileNotFoundError('Could not locate a saved raw OHLCV artifact for benchmark construction.')

    raw_df = pd.read_csv(raw_ohlcv_path)
    required_cols = {'Date', 'Ticker', 'Close'}
    if not required_cols.issubset(raw_df.columns):
        raise ValueError(f'Raw OHLCV artifact missing required columns: {required_cols - set(raw_df.columns)}')

    raw_df = raw_df.copy()
    raw_df['Date'] = pd.to_datetime(raw_df['Date'])
    raw_df['Ticker'] = raw_df['Ticker'].astype(str)
    raw_df['Close'] = pd.to_numeric(raw_df['Close'], errors='coerce')
    raw_df = raw_df.dropna(subset=['Date', 'Ticker', 'Close']).sort_values(['Ticker', 'Date']).reset_index(drop=True)
    raw_df['daily_return'] = raw_df.groupby('Ticker')['Close'].pct_change()

    test_dates = pd.Series(pd.to_datetime(test_df['Date']).dropna().unique()).sort_values().reset_index(drop=True)
    test_tickers = set(test_df['Ticker'].astype(str).dropna().unique())

    eqw_df = raw_df[raw_df['Ticker'].isin(test_tickers) & raw_df['Date'].isin(test_dates)].copy()
    eqw = eqw_df.groupby('Date')['daily_return'].mean().reindex(test_dates).astype(float)
    if eqw.isna().any():
        missing_dates = pd.Series(test_dates[eqw.isna()]).dt.strftime('%Y-%m-%d').tolist()[:5]
        raise ValueError(f'EW baseline missing returns on test dates, e.g. {missing_dates}')
    eqw = _validate_baseline_series('EW', eqw)

    spy = eval_fetch_spy_returns(test_dates.min(), test_dates.max())
    if not spy.empty:
        spy = spy.reindex(test_dates).astype(float)
        if spy.isna().any():
            missing_dates = pd.Series(test_dates[spy.isna()]).dt.strftime('%Y-%m-%d').tolist()[:5]
            raise ValueError(f'SPY baseline missing returns on test dates, e.g. {missing_dates}')
        spy = _validate_baseline_series('SPY', spy)

    baseline_meta = {
        'eqw_source': f'raw_ohlcv_close_pct::{raw_ohlcv_path.name}',
        'spy_source': 'yfinance_auto_adjusted_close_pct::SPY',
    }
    return eqw.reset_index(drop=True), (spy.reset_index(drop=True) if not spy.empty else pd.Series(dtype=float)), baseline_meta


def baseline_slice(series: pd.Series, start_offset: int, horizon_days: int) -> pd.Series:
    if series is None or len(series) == 0:
        return pd.Series(dtype=float)
    end = min(len(series), int(start_offset) + int(horizon_days))
    return pd.Series(series.iloc[int(start_offset):end]).reset_index(drop=True).astype(float)


def _append_deduped_csv(path: Path, new_df: pd.DataFrame, key_cols: list[str]) -> pd.DataFrame:
    existing = load_csv_or_empty(path)
    combined = pd.concat([existing, new_df], ignore_index=True, sort=False)
    if key_cols:
        missing = [c for c in key_cols if c not in combined.columns]
        if not missing:
            combined = combined.drop_duplicates(subset=key_cols, keep='last')
    save_df(combined, path)
    return combined


def save_baseline_track(stage_dirs: dict, block_prefix: str, label: str, returns: pd.Series) -> str | None:
    series = pd.Series(returns, dtype=float).replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    if series.empty:
        return None
    df = pd.DataFrame({
        'step': np.arange(len(series)),
        'daily_return': series.astype(float),
        'portfolio_value_index': (1.0 + series.astype(float)).cumprod(),
    })
    dst = stage_dirs['tracks'] / f'{block_prefix}__baseline_{label}.csv'
    save_df(df, dst)
    return str(dst.relative_to(EVAL_SESSION_ROOT))


def _benchmark_rows_are_current(raw_df: pd.DataFrame) -> bool:
    if raw_df is None or raw_df.empty:
        return False
    if 'benchmark_version' not in raw_df.columns:
        return False
    versions = pd.to_numeric(raw_df['benchmark_version'], errors='coerce')
    if versions.isna().any() or not (versions == int(BENCHMARK_SCHEMA_VERSION)).all():
        return False
    if 'benchmark_risk_free_rate' not in raw_df.columns:
        return False
    rf = pd.to_numeric(raw_df['benchmark_risk_free_rate'], errors='coerce')
    if rf.isna().any() or not np.isclose(rf, float(BENCHMARK_RISK_FREE_RATE)).all():
        return False
    required_cols = [
        'eqw_agent_sharpe', 'eqw_baseline_sharpe',
        'spy_agent_sharpe', 'spy_baseline_sharpe',
        'eqw_agent_total_return', 'eqw_baseline_total_return',
        'spy_agent_total_return', 'spy_baseline_total_return',
    ]
    for col in required_cols:
        if col not in raw_df.columns or raw_df[col].isna().any():
            return False
    for err_col in ['eqw_error', 'spy_error']:
        if err_col in raw_df.columns:
            err_vals = raw_df[err_col].fillna('').astype(str).str.strip()
            if (err_vals != '').any():
                return False
    return True


def ensure_benchmark_tables(stage_key: str, stage_dirs: dict) -> pd.DataFrame:
    raw_path = stage_dirs['tables'] / f'{stage_key}_raw.csv'
    raw_df = _drop_unnamed_cols(load_csv_or_empty(raw_path))
    if _benchmark_rows_are_current(raw_df):
        return raw_df
    rows = []
    for manifest_path in sorted(stage_dirs['manifests'].glob(f'{stage_key}__*.json')):
        payload = load_json_or_none(manifest_path)
        if isinstance(payload, dict) and isinstance(payload.get('benchmark_row'), dict):
            rows.append(payload['benchmark_row'])
    if not rows:
        return pd.DataFrame()
    raw_df = _drop_unnamed_cols(pd.DataFrame(rows))
    if not _benchmark_rows_are_current(raw_df):
        print(f'[INFO] {stage_key}: existing benchmark artifacts are stale or incomplete; recomputing benchmark rows.')
        return pd.DataFrame()
    if 'benchmark_id' in raw_df.columns:
        raw_df = raw_df.drop_duplicates(subset=['benchmark_id'], keep='last').sort_values('benchmark_id').reset_index(drop=True)
    save_df(raw_df, raw_path)
    return raw_df


def resolve_selected_checkpoints(df_ckpt: pd.DataFrame, selections: list[dict]) -> pd.DataFrame:
    rows = []
    for item in selections:
        episode = int(item['episode'])
        checkpoint_kind = str(item.get('checkpoint_kind', 'high_watermark'))
        ckpt_row = resolve_checkpoint_row(df_ckpt, episode=episode, checkpoint_kind=checkpoint_kind)
        label = item.get('checkpoint_label') or f'{checkpoint_kind}__ep{episode:04d}'
        row = ckpt_row.to_dict()
        row['checkpoint_label'] = label
        rows.append(row)
    out = pd.DataFrame(rows)
    return out.sort_values(['checkpoint_kind', 'episode']).reset_index(drop=True)


def run_benchmark_row(plan_row: pd.Series, checkpoint_row: pd.Series, stage_key: str, stage_dirs: dict) -> dict:
    start_offset = int(plan_row['start_offset'])
    horizon_days = int(plan_row['horizon_days'])
    phase1_slice, meta = make_phase1_slice(eval_phase1_data, start_offset, horizon_days)
    if phase1_slice is None:
        return {'benchmark_id': plan_row['benchmark_id'], 'status': 'skipped', 'reason': 'invalid_slice'}

    checkpoint_label = str(checkpoint_row['checkpoint_label'])
    checkpoint_prefix = str(checkpoint_row['checkpoint_prefix'])
    checkpoint_episode = int(checkpoint_row['episode'])
    block_prefix = f'{stage_key}__{plan_row["benchmark_id"]}__start-{meta["window_start_date"]}__h{horizon_days}__ep{checkpoint_episode:04d}'

    call_started_at = time.time()
    ev = eval_run_one_checkpoint(
        eval_config,
        phase1_slice,
        checkpoint_prefix,
        seed=int(EVAL_RANDOM_SEED + 900_000 + checkpoint_episode + horizon_days + start_offset),
        save_logs=SAVE_EVAL_LOGS,
        save_artifacts=SAVE_EVAL_ARTIFACTS,
    )
    det = ev.deterministic_metrics or {}
    agent_returns = pd.Series(np.diff(ev.deterministic_portfolio) / ev.deterministic_portfolio[:-1] if len(ev.deterministic_portfolio) > 1 else [], dtype=float)
    agent_tail = compute_tail_risk_metrics(agent_returns, alpha=CVAR_ALPHA)

    row = {
        'benchmark_id': plan_row['benchmark_id'],
        'benchmark_version': int(BENCHMARK_SCHEMA_VERSION),
        'benchmark_risk_free_rate': float(BENCHMARK_RISK_FREE_RATE),
        'checkpoint_label': checkpoint_label,
        'checkpoint_prefix': checkpoint_prefix,
        'checkpoint_kind': str(checkpoint_row['checkpoint_kind']),
        'checkpoint_episode': checkpoint_episode,
        'years': int(plan_row['years']),
        'horizon_days': horizon_days,
        'start_offset': start_offset,
        'window_start_date': meta['window_start_date'],
        'window_end_date': meta['window_end_date'],
        'eqw_source': str(EVAL_BASELINE_META.get('eqw_source', 'unknown')),
        'spy_source': str(EVAL_BASELINE_META.get('spy_source', 'unknown')),
        'det_return': float(det.get('total_return', np.nan)),
        'det_sharpe': float(det.get('sharpe_ratio', np.nan)),
        'det_mdd': float(det.get('max_drawdown_abs', det.get('max_drawdown', np.nan))),
        'det_turnover': float(det.get('turnover', np.nan)),
        'det_var_5': float(agent_tail['var_alpha']),
        'det_cvar_5': float(agent_tail['cvar_alpha']),
        'det_var_5_pct': float(agent_tail['var_alpha_pct']),
        'det_cvar_5_pct': float(agent_tail['cvar_alpha_pct']),
        'agent_total_return': float(det.get('total_return', np.nan)),
    }

    eqw_slice = baseline_slice(EVAL_BASELINE_EQW, start_offset, horizon_days)
    spy_slice = baseline_slice(EVAL_BASELINE_SPY, start_offset, horizon_days)
    eqw_tail = compute_tail_risk_metrics(eqw_slice, alpha=CVAR_ALPHA) if len(eqw_slice) > 0 else {}
    spy_tail = compute_tail_risk_metrics(spy_slice, alpha=CVAR_ALPHA) if len(spy_slice) > 0 else {}

    try:
        cmp_eqw = compare_agent_vs_baseline(ev, eqw_slice, risk_free_rate=BENCHMARK_RISK_FREE_RATE)
        for k, v in cmp_eqw.items():
            row[f'eqw_{k}'] = v
        row['eqw_agent_total_return'] = float(row['det_return'])
        row['eqw_baseline_total_return'] = float((1.0 + pd.Series(eqw_slice, dtype=float)).prod() - 1.0)
        if eqw_tail:
            row['eqw_var_5'] = float(eqw_tail['var_alpha'])
            row['eqw_cvar_5'] = float(eqw_tail['cvar_alpha'])
            row['eqw_var_5_pct'] = float(eqw_tail['var_alpha_pct'])
            row['eqw_cvar_5_pct'] = float(eqw_tail['cvar_alpha_pct'])
    except Exception as e:
        row['eqw_error'] = str(e)

    try:
        if len(spy_slice) > 0:
            cmp_spy = compare_agent_vs_baseline(ev, spy_slice, risk_free_rate=BENCHMARK_RISK_FREE_RATE)
            for k, v in cmp_spy.items():
                row[f'spy_{k}'] = v
            row['spy_agent_total_return'] = float(row['det_return'])
            row['spy_baseline_total_return'] = float((1.0 + pd.Series(spy_slice, dtype=float)).prod() - 1.0)
            row['spy_var_5'] = float(spy_tail['var_alpha'])
            row['spy_cvar_5'] = float(spy_tail['cvar_alpha'])
            row['spy_var_5_pct'] = float(spy_tail['var_alpha_pct'])
            row['spy_cvar_5_pct'] = float(spy_tail['cvar_alpha_pct'])
        else:
            row['spy_error'] = 'SPY baseline unavailable'
    except Exception as e:
        row['spy_error'] = str(e)

    raw_path = stage_dirs['tables'] / f'{stage_key}_raw.csv'
    _append_deduped_csv(raw_path, pd.DataFrame([row]), key_cols=['benchmark_id'])

    copied = copy_eval_outputs_since(call_started_at, stage_dirs, block_prefix)
    baseline_artifacts = []
    eqw_rel = save_baseline_track(stage_dirs, block_prefix, 'eqw', eqw_slice)
    spy_rel = save_baseline_track(stage_dirs, block_prefix, 'spy', spy_slice) if len(spy_slice) > 0 else None
    if eqw_rel:
        baseline_artifacts.append(eqw_rel)
    if spy_rel:
        baseline_artifacts.append(spy_rel)
    save_json({'benchmark_row': row, 'copied_artifacts': copied, 'baseline_artifacts': baseline_artifacts}, stage_dirs['manifests'] / f'{block_prefix}.json')
    return {'benchmark_id': plan_row['benchmark_id'], 'status': 'ok', 'copied_artifacts': len(copied) + len(baseline_artifacts)}


def run_benchmark_plan(plan_df: pd.DataFrame, *, stage_key: str, stage_dirs: dict, checkpoint_lookup: dict[str, pd.Series], limit_rows=None) -> pd.DataFrame:
    raw_df = ensure_benchmark_tables(stage_key, stage_dirs)
    completed = set(raw_df['benchmark_id'].dropna().astype(str)) if not raw_df.empty and 'benchmark_id' in raw_df.columns else set()
    pending = plan_df[~plan_df['benchmark_id'].astype(str).isin(completed)].copy().reset_index(drop=True)
    if limit_rows is not None:
        pending = pending.head(int(limit_rows))
    print(f'{stage_key}: completed={len(completed)} pending_now={len(pending)}')
    if pending.empty:
        print(f'[SKIP] {stage_key}: using saved outputs from {stage_dirs["root"]}')
        if not raw_df.empty:
            out = raw_df.copy()
            out.insert(0, 'status', 'reused')
            keep_cols = [c for c in ['status', 'benchmark_id', 'checkpoint_label', 'years', 'window_start_date', 'window_end_date', 'det_sharpe', 'det_return', 'det_mdd'] if c in out.columns]
            return out[keep_cols]
        return pd.DataFrame([{'status': 'reused', 'completed_benchmarks': len(completed)}])
    rows = []
    for _, row in pending.iterrows():
        benchmark_id = str(row['benchmark_id'])
        checkpoint_label = str(row['checkpoint_label'])
        print(f"[RUN] {stage_key} -> {benchmark_id} | checkpoint={checkpoint_label} | years={row['years']} | offset={row['start_offset']}")
        out = run_benchmark_row(row, checkpoint_lookup[checkpoint_label], stage_key=stage_key, stage_dirs=stage_dirs)
        rows.append(out)
        print(f"      status={out.get('status')} copied={out.get('copied_artifacts', 0)}")
    return pd.DataFrame(rows)


save_json({
    'run_id': RUN_ID,
    'session_root': str(EVAL_SESSION_ROOT),
    'results_root': str(EVAL_RESULTS_ROOT),
    'metadata_path': str(EVAL_METADATA_PATH),
    'prep_artifacts_dir': str(EVAL_PREP_ARTIFACTS_DIR),
    'benchmark_risk_free_rate': float(BENCHMARK_RISK_FREE_RATE),
    'cvar_alpha': float(CVAR_ALPHA),
    'horizons_days': HORIZON_DAYS,
    'benchmark_start_offsets': BENCHMARK_START_OFFSETS,
    'section_roots': {k: str(v['root']) for k, v in SECTION_DIRS.items()},
}, SECTION_DIRS['metadata']['manifests'] / 'session_setup.json')

print('Session root:', EVAL_SESSION_ROOT)
print('Artifact source dirs:', get_eval_artifact_sources())


## Discover and Select Checkpoints
Set `CHECKPOINT_SELECTIONS` in the config cell, then verify the resolved checkpoint rows here.


In [ ]:
eval_ckpt_df = discover_checkpoint_pairs(
    EVAL_RESULTS_ROOT,
    high_watermark_subdir=EVAL_HW_SUBDIR,
    step_sharpe_subdir=EVAL_STEP_SUBDIR,
    include_root=False,
)
selected_checkpoints_df = resolve_selected_checkpoints(eval_ckpt_df, CHECKPOINT_SELECTIONS)
checkpoint_lookup = {str(row['checkpoint_label']): row for _, row in selected_checkpoints_df.iterrows()}

save_df(eval_ckpt_df, SECTION_DIRS['metadata']['tables'] / 'checkpoint_catalog.csv')
save_df(selected_checkpoints_df, SECTION_DIRS['metadata']['tables'] / 'selected_checkpoints.csv')

display(selected_checkpoints_df)


## Build Benchmark Plan
This creates one deterministic benchmark row per `(checkpoint, horizon, start_offset)` combination.


In [ ]:
EVAL_BASELINE_EQW, EVAL_BASELINE_SPY, EVAL_BASELINE_META = build_baselines_from_phase1(eval_phase1_data)
print(EVAL_BASELINE_META)

benchmark_plan_path = SECTION_DIRS['benchmarks']['plans'] / 'benchmark_plan.csv'
if stage_file_has_rows(benchmark_plan_path, required_col='benchmark_id'):
    benchmark_plan_df = load_csv_or_empty(benchmark_plan_path)
else:
    benchmark_plan_rows = []
    for _, ckpt_row in selected_checkpoints_df.iterrows():
        checkpoint_label = str(ckpt_row['checkpoint_label'])
        checkpoint_episode = int(ckpt_row['episode'])
        for years, horizon_days in HORIZON_DAYS.items():
            for start_offset in BENCHMARK_START_OFFSETS:
                phase1_slice, meta = make_phase1_slice(eval_phase1_data, start_offset, horizon_days)
                if phase1_slice is None:
                    continue
                benchmark_plan_rows.append({
                    'benchmark_id': f'{checkpoint_label}__y{years}_off{start_offset:04d}',
                    'checkpoint_label': checkpoint_label,
                    'checkpoint_episode': checkpoint_episode,
                    'years': int(years),
                    'horizon_days': int(horizon_days),
                    'start_offset': int(start_offset),
                    'start_date': meta['window_start_date'],
                    'end_date': meta['window_end_date'],
                })
    benchmark_plan_df = pd.DataFrame(benchmark_plan_rows)
    save_df(benchmark_plan_df, benchmark_plan_path)

display(benchmark_plan_df)
print('Planned benchmark rows:', len(benchmark_plan_df))


## Run Deterministic Benchmark Sweep
This section evaluates all pending deterministic benchmark rows and saves reusable artifacts after each completed row.


In [ ]:
benchmark_run_df = run_benchmark_plan(
    benchmark_plan_df,
    stage_key='benchmarks',
    stage_dirs=SECTION_DIRS['benchmarks'],
    checkpoint_lookup=checkpoint_lookup,
    limit_rows=RUN_LIMIT_BENCHMARK,
)
benchmark_raw_df = load_csv_or_empty(SECTION_DIRS['benchmarks']['tables'] / 'benchmarks_raw.csv')

display(benchmark_run_df)
display(benchmark_raw_df.tail(20))


## Benchmark Summary by Checkpoint and Horizon
This table shows the mean deterministic benchmark metrics for each checkpoint and horizon, including explicit return columns.


In [ ]:
benchmark_raw_df = load_csv_or_empty(SECTION_DIRS['benchmarks']['tables'] / 'benchmarks_raw.csv')
if not benchmark_raw_df.empty:
    summary_agg = {
        'det_sharpe': 'mean',
        'det_return': 'mean',
        'det_mdd': 'mean',
        'det_turnover': 'mean',
        'agent_total_return': 'mean',
    }
    rename_map = {
        'det_sharpe': 'det_sharpe_mean',
        'det_return': 'det_return_mean',
        'det_mdd': 'det_mdd_mean',
        'det_turnover': 'det_turnover_mean',
        'agent_total_return': 'agent_total_return_mean',
    }
    for col in [
        'det_var_5_pct', 'det_cvar_5_pct',
        'eqw_agent_sharpe', 'eqw_baseline_sharpe',
        'eqw_agent_mean_return', 'eqw_baseline_mean_return',
        'eqw_agent_total_return', 'eqw_baseline_total_return',
        'eqw_var_5_pct', 'eqw_cvar_5_pct',
        'spy_agent_sharpe', 'spy_baseline_sharpe',
        'spy_agent_mean_return', 'spy_baseline_mean_return',
        'spy_agent_total_return', 'spy_baseline_total_return',
        'spy_var_5_pct', 'spy_cvar_5_pct',
    ]:
        if col in benchmark_raw_df.columns:
            summary_agg[col] = 'mean'
            rename_map[col] = f'{col}_mean'
    benchmark_summary_df = (
        benchmark_raw_df
        .groupby(['checkpoint_label', 'checkpoint_prefix', 'checkpoint_episode', 'years'], as_index=False)
        .agg(summary_agg)
        .rename(columns=rename_map)
        .sort_values(['checkpoint_episode', 'years'])
        .reset_index(drop=True)
    )
else:
    benchmark_summary_df = pd.DataFrame()

save_df(benchmark_summary_df, SECTION_DIRS['benchmarks']['tables'] / 'benchmark_summary.csv')
display(benchmark_summary_df)


## Cross-Horizon Outperformance Scorecard
This section identifies which selected checkpoints beat both EW and SPY across all horizons on Sharpe and return.


In [ ]:
if benchmark_summary_df.empty:
    checkpoint_horizon_scorecard_df = pd.DataFrame()
    checkpoint_overall_scorecard_df = pd.DataFrame()
    strict_winners_df = pd.DataFrame()
else:
    scorecard = benchmark_summary_df.copy()
    scorecard['beats_eqw_sharpe'] = scorecard['eqw_agent_sharpe_mean'] > scorecard['eqw_baseline_sharpe_mean']
    scorecard['beats_spy_sharpe'] = scorecard['spy_agent_sharpe_mean'] > scorecard['spy_baseline_sharpe_mean']
    scorecard['beats_both_sharpe'] = scorecard['beats_eqw_sharpe'] & scorecard['beats_spy_sharpe']
    scorecard['beats_eqw_return'] = scorecard['agent_total_return_mean'] > scorecard['eqw_baseline_total_return_mean']
    scorecard['beats_spy_return'] = scorecard['agent_total_return_mean'] > scorecard['spy_baseline_total_return_mean']
    scorecard['beats_both_return'] = scorecard['beats_eqw_return'] & scorecard['beats_spy_return']
    scorecard['eqw_sharpe_margin'] = scorecard['eqw_agent_sharpe_mean'] - scorecard['eqw_baseline_sharpe_mean']
    scorecard['spy_sharpe_margin'] = scorecard['spy_agent_sharpe_mean'] - scorecard['spy_baseline_sharpe_mean']
    scorecard['eqw_return_margin'] = scorecard['agent_total_return_mean'] - scorecard['eqw_baseline_total_return_mean']
    scorecard['spy_return_margin'] = scorecard['agent_total_return_mean'] - scorecard['spy_baseline_total_return_mean']
    scorecard['better_eqw_cvar'] = scorecard['det_cvar_5_pct_mean'] > scorecard['eqw_cvar_5_pct_mean']
    scorecard['better_spy_cvar'] = scorecard['det_cvar_5_pct_mean'] > scorecard['spy_cvar_5_pct_mean']
    checkpoint_horizon_scorecard_df = scorecard.sort_values(['checkpoint_episode', 'years']).reset_index(drop=True)

    checkpoint_overall_scorecard_df = (
        checkpoint_horizon_scorecard_df
        .groupby(['checkpoint_label', 'checkpoint_prefix', 'checkpoint_episode'], as_index=False)
        .agg(
            horizon_count=('years', 'count'),
            horizons_beating_eqw_sharpe=('beats_eqw_sharpe', 'sum'),
            horizons_beating_spy_sharpe=('beats_spy_sharpe', 'sum'),
            horizons_beating_both_sharpe=('beats_both_sharpe', 'sum'),
            horizons_beating_eqw_return=('beats_eqw_return', 'sum'),
            horizons_beating_spy_return=('beats_spy_return', 'sum'),
            horizons_beating_both_return=('beats_both_return', 'sum'),
            mean_det_sharpe=('det_sharpe_mean', 'mean'),
            min_det_sharpe=('det_sharpe_mean', 'min'),
            mean_agent_total_return=('agent_total_return_mean', 'mean'),
            min_agent_total_return=('agent_total_return_mean', 'min'),
            mean_eqw_sharpe_margin=('eqw_sharpe_margin', 'mean'),
            min_eqw_sharpe_margin=('eqw_sharpe_margin', 'min'),
            mean_spy_sharpe_margin=('spy_sharpe_margin', 'mean'),
            min_spy_sharpe_margin=('spy_sharpe_margin', 'min'),
            mean_eqw_return_margin=('eqw_return_margin', 'mean'),
            min_eqw_return_margin=('eqw_return_margin', 'min'),
            mean_spy_return_margin=('spy_return_margin', 'mean'),
            min_spy_return_margin=('spy_return_margin', 'min'),
        )
    )
    checkpoint_overall_scorecard_df['all_horizons_beat_eqw_sharpe'] = checkpoint_overall_scorecard_df['horizons_beating_eqw_sharpe'] == checkpoint_overall_scorecard_df['horizon_count']
    checkpoint_overall_scorecard_df['all_horizons_beat_spy_sharpe'] = checkpoint_overall_scorecard_df['horizons_beating_spy_sharpe'] == checkpoint_overall_scorecard_df['horizon_count']
    checkpoint_overall_scorecard_df['all_horizons_beat_both_sharpe'] = checkpoint_overall_scorecard_df['horizons_beating_both_sharpe'] == checkpoint_overall_scorecard_df['horizon_count']
    checkpoint_overall_scorecard_df['all_horizons_beat_eqw_return'] = checkpoint_overall_scorecard_df['horizons_beating_eqw_return'] == checkpoint_overall_scorecard_df['horizon_count']
    checkpoint_overall_scorecard_df['all_horizons_beat_spy_return'] = checkpoint_overall_scorecard_df['horizons_beating_spy_return'] == checkpoint_overall_scorecard_df['horizon_count']
    checkpoint_overall_scorecard_df['all_horizons_beat_both_return'] = checkpoint_overall_scorecard_df['horizons_beating_both_return'] == checkpoint_overall_scorecard_df['horizon_count']
    checkpoint_overall_scorecard_df = checkpoint_overall_scorecard_df.sort_values(
        ['all_horizons_beat_both_sharpe', 'horizons_beating_both_sharpe', 'mean_det_sharpe', 'mean_agent_total_return'],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    strict_winners_df = checkpoint_overall_scorecard_df[
        checkpoint_overall_scorecard_df['all_horizons_beat_both_sharpe'] & checkpoint_overall_scorecard_df['all_horizons_beat_both_return']
    ].copy()

save_df(checkpoint_horizon_scorecard_df, SECTION_DIRS['benchmarks']['tables'] / 'checkpoint_horizon_scorecard.csv')
save_df(checkpoint_overall_scorecard_df, SECTION_DIRS['benchmarks']['tables'] / 'checkpoint_overall_scorecard.csv')
save_df(strict_winners_df, SECTION_DIRS['benchmarks']['tables'] / 'checkpoint_strict_winners.csv')

display(checkpoint_horizon_scorecard_df)
display(checkpoint_overall_scorecard_df)
display(strict_winners_df)


## Save Manifest
This records the benchmark sweep inputs and outputs for later reuse.


In [ ]:
benchmark_manifest = {
    'run_id': RUN_ID,
    'session_tag': SESSION_TAG,
    'session_root': str(EVAL_SESSION_ROOT),
    'results_root': str(EVAL_RESULTS_ROOT),
    'metadata_path': str(EVAL_METADATA_PATH),
    'selected_checkpoints': selected_checkpoints_df[['checkpoint_label', 'checkpoint_kind', 'episode', 'checkpoint_prefix']].to_dict(orient='records'),
    'benchmark_risk_free_rate': float(BENCHMARK_RISK_FREE_RATE),
    'cvar_alpha': float(CVAR_ALPHA),
    'horizons_days': HORIZON_DAYS,
    'benchmark_start_offsets': BENCHMARK_START_OFFSETS,
    'asset_tickers': list(eval_config['ASSET_TICKERS']),
    'tables': {
        'selected_checkpoints': str(SECTION_DIRS['metadata']['tables'] / 'selected_checkpoints.csv'),
        'benchmark_plan': str(SECTION_DIRS['benchmarks']['plans'] / 'benchmark_plan.csv'),
        'benchmark_raw': str(SECTION_DIRS['benchmarks']['tables'] / 'benchmarks_raw.csv'),
        'benchmark_summary': str(SECTION_DIRS['benchmarks']['tables'] / 'benchmark_summary.csv'),
        'checkpoint_horizon_scorecard': str(SECTION_DIRS['benchmarks']['tables'] / 'checkpoint_horizon_scorecard.csv'),
        'checkpoint_overall_scorecard': str(SECTION_DIRS['benchmarks']['tables'] / 'checkpoint_overall_scorecard.csv'),
        'checkpoint_strict_winners': str(SECTION_DIRS['benchmarks']['tables'] / 'checkpoint_strict_winners.csv'),
    },
}
save_json(benchmark_manifest, SECTION_DIRS['manifest']['manifests'] / 'benchmark_manifest.json')
benchmark_manifest
